# Outcome invariance

Check the eleven paper outcomes under behavioral prompt versions 0 and 2
Assess variation across profiles within each housing condition

Run from the project root or analysis folder


## Prompt versions

| Version | Wording |
| --- | --- |
| 0 | Original |
| 2 | Broad paraphrase |


In [ ]:
from pathlib import Path
import os
import sys

ROOT = next((candidate for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent)
             if (candidate / "analysis" / "analysis_utils.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the project root or its analysis folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Resolve inputs from the project root
os.chdir(ROOT)

import hashlib
import numpy as np
import pandas as pd
from analysis.analysis_utils import DATA_DIR, RESULT_DIR, MODELS

RAW_HASHES_BEFORE = {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
                     for path in DATA_DIR.iterdir() if path.suffix in {".csv", ".jsonl"}}
EXPECTED_PROFILES = 224
PROMPT_VERSIONS = [0, 2]
NUMERIC_MEASURES = ["Overdose", "Public_prop", "Recept_sharing", "Partners", "Drug_in", "Drug_out",
                    "Partner_HCV_prop", "Treatment", "Injections", "Alt_route", "HCV_active"]
MEASURES = NUMERIC_MEASURES
GROUPS = ["Model", "Paraphrase", "Homeless"]


In [ ]:
data = pd.read_csv(DATA_DIR / "outputs.csv")
data = data.loc[data.Paraphrase.isin(PROMPT_VERSIONS)].copy()
keys = GROUPS + ["Agent"]
if data[keys].isna().any().any() or data.duplicated(keys).any():
    raise ValueError("Profile identifiers must be unique and nonmissing within each model/prompt/housing group.")
expected_groups = {(model, version, housing) for model in MODELS for version in PROMPT_VERSIONS for housing in [0, 1]}
if set(data.groupby(GROUPS).groups) != expected_groups:
    raise ValueError("Expected all three models, two retained prompt versions, and two housing conditions.")
cohort = None
for group_key, group in data.groupby(GROUPS):
    agents = set(group.Agent)
    if len(agents) != EXPECTED_PROFILES or (cohort is not None and agents != cohort):
        raise ValueError(f"{group_key}: expected the same 224 profiles in every group.")
    cohort = agents
hcv = data.HCV.astype("string").str.strip().str.lower()
if not hcv.dropna().isin(["susceptible", "acute"]).all():
    raise ValueError("Unexpected HCV state.")
data["HCV_active"] = hcv.map({"susceptible": 0, "acute": 1}).astype(float)
for measure in NUMERIC_MEASURES:
    data[measure] = pd.to_numeric(data[measure], errors="raise")
    if np.isinf(data[measure]).any() or (data[measure].dropna() < 0).any():
        raise ValueError(f"Invalid numeric output: {measure}")
for measure in ["Public_prop", "Recept_sharing", "Partner_HCV_prop", "Treatment", "HCV_active"]:
    if data[measure].dropna().gt(1).any():
        raise ValueError(f"Out-of-range proportion/binary output: {measure}")
if not data.Treatment.dropna().isin([0, 1]).all():
    raise ValueError("Treatment must be binary.")
print(f"Loaded {len(data):,} saved responses: {len(expected_groups)} groups of {EXPECTED_PROFILES} profiles.")


## Within-condition invariance

An invariant outcome has one distinct value among at least two observed profiles
Count missing values separately


In [ ]:
def audit_invariance(frame, measures):
    if frame[GROUPS + ["Agent"]].isna().any().any() or frame.duplicated(GROUPS + ["Agent"]).any():
        raise ValueError("Duplicate or missing profile identifiers.")
    records = []
    for group_key, group in frame.groupby(GROUPS, sort=True):
        group = group.sort_values("Agent")
        for measure in measures:
            observed = group[measure].dropna()
            n = len(observed)
            distinct = int(observed.nunique())
            invariant = n >= 2 and distinct == 1
            counts = observed.value_counts()
            numeric = pd.api.types.is_numeric_dtype(group[measure])
            status = ("all_missing" if n == 0 else "insufficient_data" if n == 1
                      else "invariant" if invariant else "variable")
            records.append(dict(zip(GROUPS, group_key)) | dict(
                Housing="Stable" if group_key[2] == 0 else "Homeless", Measure=measure,
                outcome_type="binary" if measure in ["Treatment", "HCV_active"] else "numeric" if numeric else "categorical",
                N_profiles=len(group), N_nonmissing=n, N_missing=len(group)-n, N_unique=distinct,
                invariant=bool(invariant), status=status,
                constant_value=observed.iloc[0] if invariant else None,
                modal_value=counts.index[0] if n else None,
                modal_fraction=float(counts.iloc[0]/n) if n else np.nan,
                minimum=float(observed.min()) if numeric and n else np.nan,
                maximum=float(observed.max()) if numeric and n else np.nan))
    return pd.DataFrame(records)

invariance_audit = audit_invariance(data, MEASURES)
assert len(invariance_audit) == len(MODELS)*len(PROMPT_VERSIONS)*2*len(MEASURES)


## Paired housing differences

A constant paired difference is the same across complete profiles
A zero paired difference means every complete profile has no housing change
Require at least two complete pairs


In [ ]:
def audit_paired_differences(frame, measures):
    if frame.duplicated(GROUPS + ["Agent"]).any():
        raise ValueError("Duplicate profile identifiers.")
    records = []
    for (model, version), group in frame.groupby(["Model", "Paraphrase"], sort=True):
        for measure in measures:
            wide = group.pivot(index="Agent", columns="Homeless", values=measure).reindex(columns=[0, 1])
            complete = wide.replace([np.inf, -np.inf], np.nan).dropna()
            delta = complete[1] - complete[0]
            n = len(delta)
            distinct = int(delta.nunique())
            constant = n >= 2 and distinct == 1
            by_condition = {condition: wide[condition].dropna() for condition in [0, 1]}
            flags = {condition: len(values) >= 2 and values.nunique() == 1 for condition, values in by_condition.items()}
            records.append(dict(Model=model, Paraphrase=version, Measure=measure,
                N_complete_pairs=n, N_excluded=len(wide)-n, N_unique_differences=distinct,
                N_increasing=int(delta.gt(0).sum()), N_decreasing=int(delta.lt(0).sum()),
                N_unchanged=int(delta.eq(0).sum()), sum_paired_differences=float(delta.sum()) if n else np.nan,
                stable_invariant=bool(flags[0]), homeless_invariant=bool(flags[1]),
                constant_paired_difference=bool(constant),
                zero_paired_difference=bool(constant and delta.iloc[0] == 0),
                constant_difference=float(delta.iloc[0]) if constant else np.nan,
                mean_paired_difference=float(delta.mean()) if n else np.nan,
                status="no_complete_pairs" if n == 0 else "insufficient_data" if n == 1 else "constant_difference" if constant else "varying_difference"))
    return pd.DataFrame(records)

paired_differences = audit_paired_differences(data, NUMERIC_MEASURES)
assert len(paired_differences) == len(MODELS)*len(PROMPT_VERSIONS)*len(NUMERIC_MEASURES)


## Invariance results

Save 132 checks across models, prompt versions, housing conditions and outcomes


In [ ]:
summary = invariance_audit.groupby(GROUPS + ["Housing"], sort=True).agg(
    N_profiles=("N_profiles", "first"), N_measures=("Measure", "size"),
    N_invariant=("invariant", "sum"),
    N_variable=("status", lambda values: int(values.eq("variable").sum())),
    N_all_missing=("status", lambda values: int(values.eq("all_missing").sum())),
    N_insufficient=("status", lambda values: int(values.eq("insufficient_data").sum())),
    N_measures_with_missing=("N_missing", lambda values: int(values.gt(0).sum()))).reset_index()
invariant_names = invariance_audit.loc[invariance_audit.invariant].groupby(GROUPS).Measure.agg("; ".join)
summary = summary.merge(invariant_names.rename("invariant_measures"), on=GROUPS, how="left")
summary["invariant_measures"] = summary.invariant_measures.fillna("")
assert len(summary) == len(MODELS)*len(PROMPT_VERSIONS)*2
RESULT_DIR.mkdir(exist_ok=True)
invariance_audit.to_csv(RESULT_DIR / "invariance_audit.csv", index=False)
print("Original prompt: invariant outcomes within each housing condition")
print(invariance_audit.loc[invariance_audit.Paraphrase.eq(0) & invariance_audit.invariant,
      ["Model", "Housing", "Measure", "N_nonmissing", "constant_value"]].to_string(index=False))
print("\nAll prompt versions: within-condition summary")
print(summary.to_string(index=False))


## Zero mean housing differences

- The same constant outcome under both conditions
- Outcomes vary across profiles but have no paired change
- Positive and negative paired changes cancel

Use complete pairs and unrounded values


In [ ]:
ZERO_PATTERNS = ["Same constant in both conditions", "Profiles vary; no paired change", "Opposing changes cancel"]

def classify_zero_means(paired):
    result = paired.copy()
    labels = []
    for row in result.itertuples():
        if row.N_complete_pairs < 2 or not np.isfinite(row.mean_paired_difference):
            label = "Insufficient complete pairs"
        elif row.mean_paired_difference != 0:
            label = "Nonzero mean"
        elif row.zero_paired_difference:
            label = ZERO_PATTERNS[0] if row.stable_invariant and row.homeless_invariant else ZERO_PATTERNS[1]
        else:
            label = ZERO_PATTERNS[2]
        labels.append(label)
    result["zero_mean_pattern"] = labels
    result["zero_mean"] = result.zero_mean_pattern.isin(ZERO_PATTERNS)
    return result

paired_differences = classify_zero_means(paired_differences)
breakdown_rows = []
for version, group in paired_differences.groupby("Paraphrase", sort=True):
    counts = group.zero_mean_pattern.value_counts()
    breakdown_rows.append(dict(Prompt=f"Paraphrase {version}", N_comparisons=len(group),
        N_zero_mean=int(group.zero_mean.sum()),
        **{label: int(counts.get(label, 0)) for label in ZERO_PATTERNS}))
counts = paired_differences.zero_mean_pattern.value_counts()
breakdown_rows.append(dict(Prompt="All versions", N_comparisons=len(paired_differences),
    N_zero_mean=int(paired_differences.zero_mean.sum()),
    **{label: int(counts.get(label, 0)) for label in ZERO_PATTERNS}))
zero_mean_breakdown = pd.DataFrame(breakdown_rows)
print("Zero mean paired differences: diagnostic breakdown")
print(zero_mean_breakdown.to_string(index=False))
print("\nOriginal-prompt zero-mean cases")
print(paired_differences.loc[paired_differences.Paraphrase.eq(0) & paired_differences.zero_mean,
      ["Model", "Measure", "N_complete_pairs", "N_excluded", "zero_mean_pattern"]].to_string(index=False))


## Canceling HCV differences

Compare susceptible and acute predictions across housing conditions
These are scenario predictions, not disease transitions


In [ ]:
cancellation_cases = paired_differences.loc[paired_differences.zero_mean_pattern.eq(ZERO_PATTERNS[2])].copy()
print("Zero-mean cancellation cases across all prompt versions")
print(cancellation_cases[["Model", "Paraphrase", "Measure", "N_complete_pairs", "N_excluded",
      "N_increasing", "N_decreasing", "N_unchanged", "mean_paired_difference"]].to_string(index=False))

case = data.loc[data.Model.eq("gpt-oss-120b") & data.Paraphrase.eq(2)]
case_wide = case.pivot(index="Agent", columns="Homeless", values="HCV_active").reindex(columns=[0, 1])
case_complete = case_wide.dropna()
case_delta = case_complete[1] - case_complete[0]
case_counts = []
for stable, homeless in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    case_counts.append(dict(Stable="susceptible" if stable == 0 else "acute",
        Homeless="susceptible" if homeless == 0 else "acute",
        N_profiles=int((case_complete[0].eq(stable) & case_complete[1].eq(homeless)).sum()),
        Paired_difference=homeless-stable))
hcv_cancellation_counts = pd.DataFrame(case_counts)
case_mean = float(case_delta.mean()) if len(case_delta) else np.nan
case_is_cancellation = bool(len(case_delta) >= 2 and case_mean == 0 and case_delta.gt(0).any() and case_delta.lt(0).any())
print("\nGPT-OSS-120B, Paraphrase 2: paired HCV classifications")
print(hcv_cancellation_counts.to_string(index=False))
print(f"Complete pairs: {len(case_complete)}; excluded for missing HCV status: {len(case_wide)-len(case_complete)}.")
print(f"Mean paired difference: {100*case_mean:.2f} percentage points.")
print(f"Profiles with different HCV classifications across housing conditions: {int(case_delta.ne(0).sum())} "
      f"of {len(case_delta)} ({100*case_delta.ne(0).mean():.2f}%).")
print(f"Opposing changes cancel: {case_is_cancellation}.")


## Paired diagnostic results


In [ ]:
print(paired_differences[["Model", "Paraphrase", "Measure", "N_complete_pairs", "N_excluded",
      "stable_invariant", "homeless_invariant", "constant_paired_difference", "zero_paired_difference",
      "N_increasing", "N_decreasing", "N_unchanged", "mean_paired_difference", "zero_mean_pattern"]].to_string(index=False))
RAW_HASHES_AFTER = {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
                    for path in DATA_DIR.iterdir() if path.suffix in {".csv", ".jsonl"}}
assert RAW_HASHES_BEFORE == RAW_HASHES_AFTER, "Raw CSV/JSONL contents changed."
print(f"\nSaved only {RESULT_DIR / 'invariance_audit.csv'}. Summaries and cancellation details are notebook outputs.")
print(f"All {len(RAW_HASHES_AFTER)} raw CSV/JSONL files are unchanged.")
